# DiagnosticIQ test rules

Converts the hand-written test rules in `$ASSETOPSBENCH_DATASET_DIR/test_datasets/rule_logic_test2.json` into `test_datasets/TEST_rule_information.json` and `test_datasets/TEST_TreeStruct.pkl`, the inputs to `diagnosticiq_simple_test_dataset_creation.ipynb`.

In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any, Tuple, Dict
from abc import abstractmethod, ABC
import random
import numpy as np
import pandas as pd
import copy
import pickle
import itertools
from dataclasses import dataclass, asdict
from tqdm import tqdm
from enum import Enum



In [ ]:
from dataset_utils.paths import get_dataset_dir
from dataset_utils.utils import file_handle

test_data_dir = get_dataset_dir() / "test_datasets"


In [ ]:
base_rule_info = file_handle.load_json(test_data_dir / "rule_logic_test2.json")

In [ ]:
base_rule_info['AC00003B?'],

In [ ]:
from dataset_utils.utils.tree import Node, print_tree


def dict_value_string_function(d:dict, func:Callable[[Optional[Any]], Optional[Any]]) -> dict:
    r_dict = {}
    for k,v in d.items():
        if isinstance(v, dict):
            r_dict[k] = dict_value_string_function(v,func)
        else:
            r_dict[k] = func(v)

    return r_dict

class OPS:
    ops = ["OR", "AND"]

    def __init__(self) -> None:
        for o in self.ops:
            setattr(self,o,o)



def common_string_replace(string:str):
    return string.replace("\n", " ").replace("AND/OR", "OR")


def process_rule(rule_str:str) -> Node:
    # break rule list
    individual_rules = list(map(lambda x: x.strip(),rule_str.split("\n -")))
    individual_rules[0] = individual_rules[0].replace("- ", "")

    #print(individual_rules)

    #extract conditionals
    cond_info_rules = []
    for i in range(1,len(individual_rules)):
        r1 = common_string_replace(individual_rules[i-1])
        r2 = common_string_replace(individual_rules[i])

        #print(r1,"$$$")
        #print(r2, "$$$$")

        try:
            _cond_r1 = "OR" in r1[:4]
        except IndexError:
            _cond_r1 = False

        try:
            _cond_r2 = "OR" in r2[:4]
        except IndexError:
            _cond_r2 = False
            

        if  _cond_r1 and _cond_r2:
            cond_info_rules[-1].append(r1[2:].strip())
        elif _cond_r2:
            #cond_info_rules.append([r1,r2[2:].strip()])
            cond_info_rules.append([r1])
        elif _cond_r1:
            cond_info_rules[-1].append(r1[2:].strip())
        else:
            cond_info_rules.append(r1)

        if i == len(individual_rules) - 1 :
            if _cond_r2:
                cond_info_rules[-1].append(r2[2:].strip())
            else:
                cond_info_rules.append(r2)

        #print(cond_info_rules, "****")

    
    full_root = Node('AND', rule_str, True)
    full_root.children = []
    full_root.val = "root"
    for c in cond_info_rules:
        if isinstance(c,str):
            full_root.children.append(Node.generate_graph(c))
        else:
            _temp = Node('OR', None, True)
            _temp.children = [Node.generate_graph(x) for x in c]
            _s = " OR ".join([x.statement for x in _temp.children])
            _temp.statement = _s
            _temp.val = 'OR'
            full_root.children.append(_temp)

    return full_root



In [ ]:
rules_conv_dict = {
    "date":str(datetime.now()),
    "rule_set":[]
}

for i, (k,v) in enumerate(base_rule_info.items()):
    display_text = {
        'observations':v['observations'],
        'rules':" \n ".join(["- "+x for x in v['conditions']+[v["temporal"]]]),
        'conditions':v['conditions'],
        "date":str(datetime.now()),
        'temporal':v['temporal']
    }
    rules_conv_dict['rule_set'].append({
        "#n": i,
        "id":"TEST_RULE_"+k,
        "name": v['rule_name'],
        'asset':v['asset'],
        'display_text':display_text,
    })

In [ ]:
rules_conv_dict

In [ ]:
data = rules_conv_dict

In [ ]:
file_handle.save_json(rules_conv_dict, test_data_dir / "TEST_rule_information.json")

In [ ]:
processed_data = copy.deepcopy(data)
for pd, d in zip(processed_data["rule_set"],data["rule_set"]):
    pd["display_text"]["rules"] = process_rule(d["display_text"]["rules"])

In [ ]:
processed_data

In [ ]:
for i,pd in enumerate(processed_data["rule_set"]):
    print(f"Asset:{pd['asset']}")
    print("Rules:")
    print_tree(pd["display_text"]["rules"])
    print("Observations:")
    print(pd["display_text"]["observations"])
    print("\n")
    

In [ ]:
file_handle.save_pickle(
    processed_data,
    test_data_dir / "TEST_TreeStruct.pkl"
)